# 파이썬으로 저장, 조회

SQLEditor에서 테스트한 SQL문 - insert, update, delete, select 문장을 코드로 실행한다.

In [ ]:
# Supabase 연결
from db import supabase

users_result = supabase.table('users').select('*', count='exact').execute()
users_result.count

In [76]:
#수퍼베이스 연결 모듈 호출 (db.py)
from db import supabase

In [ ]:
result = supabase.table('users').select('*', count='exact').execute()
print(result.count)

In [104]:
# Python에서 여러 사용자를 추가한다.
# 이미 같은 이메일이 있으면 수정하고, 없으면 새로 추가한다.
result = supabase.table("users").upsert([
    {"email": "kim@example.com", "username": "김철수"},
    {"email": "lee@example.com", "username": "이영희"},
    {"email": "park@example.com", "username": "박민수"},
    {"email": "choi@example.com", "username": "최지은"},
    {"email": "jung@example.com", "username": "정하늘"},
], on_conflict="email").execute()
print("처리된 사용자 수:", len(result.data))

처리된 사용자 수: 5


In [105]:
# 사용자 추가
# 노트북에서 셀을 다시 실행해도 중복 이메일 오류가 나지 않도록 먼저 확인한다.
email = "sql_to_py@example.com"
existing_result = supabase.table("users").select("*").eq("email", email).execute()

if existing_result.data:
    new_user = existing_result.data[0]
    print("이미 존재하는 사용자:", new_user)
else:
    insert_result = supabase.table("users").insert({
        "email": email,
        "username": "파이썬",
    }).execute()
    new_user = insert_result.data[0]
    print("추가된 사용자:", new_user)

new_user_id = new_user["id"]

이미 존재하는 사용자: {'id': 'cea6d875-1019-4f08-a40d-6deb4090e57e', 'email': 'sql_to_py@example.com', 'username': '박영희', 'created_at': '2026-08-24T03:28:02.764773+00:00'}


In [106]:
# 전체 조회
result = supabase.table("users").select("username, email").execute()
print("전체 목록:")
for user in result.data:
    print("   ", user["username"], "|", user["email"])

# 조건 조회
result = supabase.table("users").select("*").eq("email", "sql_to_py@example.com").execute()
print("조건으로 찾기:", result.data)

# 정렬 + 개수 제한
result = supabase.table("users").select("username").order("created_at", desc=True).limit(3).execute()
print("최근 가입 3명:")
for user in result.data:
    print("   ", user["username"])

전체 목록:
    박영희 | sql_to_py@example.com
    김철수 | kim@example.com
    이영희 | lee@example.com
    박민수 | park@example.com
    최지은 | choi@example.com
    정하늘 | jung@example.com
    테스터 | tester@example.com
    최길동 | a@example.com
    유길동 | b@example.com
조건으로 찾기: [{'id': 'cea6d875-1019-4f08-a40d-6deb4090e57e', 'email': 'sql_to_py@example.com', 'username': '박영희', 'created_at': '2026-08-24T03:28:02.764773+00:00'}]
최근 가입 3명:
    박영희
    이영희
    김철수


In [107]:
# 대화 1건 추가
conversation_result = supabase.table("conversations").insert({
    "user_id": new_user_id,
    "title": "파이썬으로 만든 대화",
}).execute()


In [108]:
conversation = conversation_result.data[0]
conversation_id = conversation["id"]
print("대화 생성:", conversation["title"])


대화 생성: 파이썬으로 만든 대화


In [109]:

# 메시지 2건을 한 번에. 여러 건은 리스트로 넘긴다.
result = supabase.table("messages").insert([
    {"conversation_id": conversation_id, "role": "user", "content": "파이썬에서도 저장이 되나요?"},
    {"conversation_id": conversation_id, "role": "assistant", "content": "네, 방금 저장됐습니다."},
]).execute()

print("메시지", len(result.data), "건 저장")

메시지 2 건 저장


In [110]:
result = supabase.table("users").select("*").eq("email", "sql_to_py@example.com").execute()
print("조건으로 찾기:", result.data)

조건으로 찾기: [{'id': 'cea6d875-1019-4f08-a40d-6deb4090e57e', 'email': 'sql_to_py@example.com', 'username': '박영희', 'created_at': '2026-08-24T03:28:02.764773+00:00'}]


In [111]:
# 조회 결과에서 첫 번째 사용자 확인
user_result = supabase.table("users").select("*").eq("email", "tester@example.com").execute()
user_result.data[0] if user_result.data else "사용자를 찾지 못했습니다."

{'id': '2efbb2d9-4952-4da7-bb89-5e332e7bee4b',
 'email': 'tester@example.com',
 'username': '테스터',
 'created_at': '2026-08-21T05:28:57.804707+00:00'}

In [112]:
# LEFT JOIN 에 해당
result = supabase.table("users").select("username, conversations(title)").execute()

for user in result.data:
    print(" ", user["username"])
    if len(user["conversations"]) == 0:
        print("     (대화 없음)")
    for conversation in user["conversations"]:
        print("    -", conversation["title"])

# INNER JOIN 에 해당
result = supabase.table("users").select("username, conversations!inner(title)").execute()
print("!inner 를 쓰면:")
for user in result.data:
    print("   ", user["username"])

# 3단계 중첩
result = (
    supabase.table("users")
    .select("username, conversations(title, messages(role, content))")
    .eq("email", "sql_to_py@example.com")
    .execute()
)

for user in result.data:
    for conversation in user["conversations"]:
        print(" ", user["username"], "-", conversation["title"])
        for message in conversation["messages"]:
            print("    ", message["role"], ":", message["content"])

  박영희
    - 파이썬으로 만든 대화
    - 파이썬으로 만든 대화
    - 파이썬으로 만든 대화
  김철수
    - 파이썬 기초 질문
    - 이직 고민 상담
  이영희
    - SQL 공부 방법
  박민수
    - 여행 계획 짜기
  최지은
     (대화 없음)
  정하늘
     (대화 없음)
  테스터
    - 유령 사용자의 대화
  최길동
     (대화 없음)
  유길동
     (대화 없음)
!inner 를 쓰면:
    박영희
    김철수
    이영희
    박민수
    테스터
  박영희 - 파이썬으로 만든 대화
     user : 파이썬에서도 저장이 되나요?
     assistant : 네, 방금 저장됐습니다.
  박영희 - 파이썬으로 만든 대화
     user : 파이썬에서도 저장이 되나요?
     assistant : 네, 방금 저장됐습니다.
     user : 파이썬에서도 저장이 되나요?
     assistant : 네, 방금 저장됐습니다.
     user : 파이썬에서도 저장이 되나요?
     assistant : 네, 방금 저장됐습니다.
  박영희 - 파이썬으로 만든 대화
     user : 파이썬에서도 저장이 되나요?
     assistant : 네, 방금 저장됐습니다.
